# Mitotic cell division simulation


## How mitotic-spindle cell division works

In a real epithelium a dividing cell does not split along an arbitrary line: the
orientation of the cleavage plane is set by the **mitotic spindle**, a bipolar
microtubule structure that orients itself inside the cell before cytokinesis. This
notebook reproduces that mechanism on a VertAX `BoundedMesh` in three stages — find the
spindle, cut along it, then relax the new tissue — and then chains them into a lineage of
successive divisions.

### 1. Finding the spindle — `spindle_model.find_spindle`

The spindle is modelled as **two centrosomes** held a fixed distance apart (a fraction
`d_fraction` of the cell's mean radius), connected to the cell cortex by **microtubules**.
Each microtubule pulls with a force proportional to the local **edge tension**, so
high-tension junctions pull harder; the cortical pull is integrated along every edge with
Gauss–Legendre quadrature (`n_gauss` points, via `_centrosome_force_jax`). The spindle
settles where the **net force and net torque on the two centrosomes vanish**:

$$F_1 + F_2 = 0, \qquad M_1 + M_2 = 0.$$

`find_spindle` returns the spindle **centre** `c` and **axis** `u`. Three strategies are
available through `method`:

- **`'newton'`** (default) — Newton–Raphson on the full force/torque balance;
  deterministic, and the centre may drift from the centroid.
- **`'pdf'`** — treats orientation as a Boltzmann-like density
  $p(\alpha)\propto\exp(-\bar V(\alpha)/C)$ and takes its **mode** (most probable angle).
- **`'pdf_sample'`** — draws a **random** orientation from that same density, modelling the
  experimental variability; `C` sets the noise level.

The microtubule density can additionally be biased by **cell polarity**
$\Lambda(\theta)=\sigma_j\,(1+p\cos^2(\theta-\varphi))$, so a polarised cell preferentially
orients its spindle along its polarity axis $\varphi$. (The PDF and polarity variants are
detailed in their own sections further down.)

### 2. Cutting the cell — `cell_division.divide_cell`

Given the spindle axis `u`, `divide_cell(mesh, face_id, c, u)` performs the **half-edge
topology surgery**: it intersects the cleavage line (perpendicular to the spindle axis,
as in real cytokinesis) with the cell boundary, inserts two new vertices at the crossings
plus a new edge between them, and rewires the half-edge table so one cell becomes two. It
returns the new mesh (one extra face) and the index of the new daughter cell.

### 3. Relaxing the tissue — `energy_adaptive` + `BoundedBilevelOptimizer`

Just after the cut the daughter cells are not mechanically balanced. We relax the mesh
with the standard bilevel optimizer (`inner_optimization`), minimising `energy_adaptive`
— which drives every cell toward a shared **target area** (total area ÷ number of cells)
while tension shortens edges — with **T1 transitions enabled** so neighbour exchanges can
occur as the geometry settles.

Repeating these three steps yields a **lineage of divisions**; the sections below run 5
consecutive divisions with the deterministic, PDF, stochastic, and polarity-biased
spindle variants.

In [ ]:
import importlib
import sys
sys.path.insert(0, '/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle')
import os
# os.environ.setdefault('JAX_PLATFORMS', 'cpu')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('JAX_ENABLE_X64', 'true')

# Run on GPU: jax[cuda12] is installed, but jax 0.4.35 crashes auto-detecting the
# CUDA path because the nvidia-cuda-nvcc-cu12 wheel is a namespace package
# (cuda_nvcc.__file__ is None). Setting CUDA_ROOT makes jax use it directly and
# skip the broken nvcc import. Points at the pip-installed nvcc dir (has bin/ptxas
# and nvvm/libdevice).
import sysconfig
_nvcc = os.path.join(sysconfig.get_paths()['purelib'], 'nvidia', 'cuda_nvcc')
if os.path.isdir(_nvcc):
    os.environ.setdefault('CUDA_ROOT', _nvcc)

# The inline backend crops figures with bbox_inches='tight' by default, which
# leaves a blank band above figures that use manually-placed axes (e.g. the
# side-by-side spindle plot). Disable that cropping so figures render as laid out.
%config InlineBackend.print_figure_kwargs = {'bbox_inches': None}

import optax
import jax
print(jax.devices())
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from vertax import BoundedMesh, BoundedBilevelOptimizer, plot_mesh, get_plot_mesh
from vertax import EdgePlot, FacePlot, VertexPlot
from vertax.topo import update_T1_bounded

from mesh_utils import get_cell_segments
from spindle_model import find_spindle
from visualization import _plot_spindle_fig, _plot_spindle_pair, _plot_post_div_fig, _plot_opt_fig

import energy_adaptive; importlib.reload(energy_adaptive)
from energy_adaptive import energy_adaptive
import cell_division; importlib.reload(cell_division)
from cell_division import divide_cell
import visualization; importlib.reload(visualization)
from visualization import _plot_spindle_fig, _plot_spindle_pair, _plot_post_div_fig, _plot_opt_fig

In [ ]:
import math

n_cells = 20
width = math.sqrt(n_cells)
height = width

mesh = BoundedMesh.from_random_seeds(nb_seeds=n_cells, width=width, height=height, random_key=2)


## Initial mesh and relaxation

Set per-element parameters (edge tensions, target areas) on the freshly seeded mesh, then
relax it with the bilevel optimizer using an **SGD** inner solver. The mesh is snapshotted
just before relaxation so the result can be shown **before / after** side by side.

In [ ]:
mesh.vertices_params = jnp.asarray([0.0 for _ in range(mesh.nb_vertices)])

key = jax.random.PRNGKey(3)
mesh.edges_params = 1 + jax.nn.sigmoid(jax.random.uniform(key, (mesh.nb_edges,)) * 20 - 10)

mesh.faces_params = jnp.asarray([0.6 for _ in range(mesh.nb_faces)])


In [ ]:
# Snapshot of the mesh BEFORE relaxation (for the before/after comparison plot).
mesh_initial = BoundedMesh.copy_mesh(mesh)

# Fix the common 'Area of cell' colour scale (used by every mesh plot in the
# notebook) to the raw initial mesh's area range.
import visualization
_init_areas = mesh_initial.get_area(jnp.arange(mesh_initial.nb_faces))
visualization.set_area_scale(float(_init_areas.min()), float(_init_areas.max()))

bilevel_optimizer = BoundedBilevelOptimizer()
bilevel_optimizer.loss_function_inner = energy_adaptive
bilevel_optimizer.inner_solver        = optax.sgd(learning_rate=1e-3)
bilevel_optimizer.update_T1           = True
bilevel_optimizer._update_T1_func     = update_T1_bounded
bilevel_optimizer.min_dist_T1         = 0.01
bilevel_optimizer.max_nb_iterations   = 1000
bilevel_optimizer.tolerance           = 1e-6
bilevel_optimizer.patience            = 50

loss_history = bilevel_optimizer.inner_optimization(mesh)

In [ ]:
def _render_mesh(snap, title, ax):
    """Render a mesh (tension on edges, area on cells) into an existing axis."""
    m = BoundedMesh.copy_mesh(snap)
    m.angles = jnp.clip(m.angles, 0.017, jnp.pi / 2 - 0.01)
    fig_tmp, _ = get_plot_mesh(m,
                               edge_plot=EdgePlot.EDGE_PARAMETER, edge_parameters_name="tension",
                               face_plot=FacePlot.AREA,
                               vertex_plot=VertexPlot.INVISIBLE)
    fig_tmp.canvas.draw()
    w, h = fig_tmp.canvas.get_width_height()
    img = np.frombuffer(fig_tmp.canvas.tostring_argb(), dtype=np.uint8).reshape(h, w, 4)
    img = np.roll(img, -1, axis=2)  # ARGB -> RGBA
    plt.close(fig_tmp)
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis("off")


area_before = float(jnp.sum(mesh_initial.get_area(jnp.arange(mesh_initial.nb_faces))))
area_after  = float(jnp.sum(mesh.get_area(jnp.arange(mesh.nb_faces))))

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("Initial relaxation (SGD)", fontsize=13)
_render_mesh(mesh_initial, f"Before relaxation — total area: {area_before:.3f}", axes[0])
_render_mesh(mesh,         f"After relaxation ({len(loss_history)} it) — total area: {area_after:.3f}", axes[1])
plt.tight_layout()
plt.show()


## Mitotic spindle


In [ ]:
import importlib
import spindle_model; importlib.reload(spindle_model)
import cell_division; importlib.reload(cell_division)
from spindle_model import find_spindle
from cell_division import divide_cell

# Find a divisible cell (spindle converges + exactly 2 cuts)
import numpy as np
rng = np.random.default_rng(0)
face_id = None
c = u = c1 = c2 = v0s = v1s = None
for fid in rng.permutation(mesh.nb_faces).tolist():
    try:
        from mesh_utils import get_cell_segments
        v0s_try, v1s_try, tensions_try = get_cell_segments(fid, mesh)
        c_try, u_try, c1_try, c2_try = find_spindle(fid, mesh, d_fraction=0.3, n_gauss=12)
        mesh_test = BoundedMesh.copy_mesh(mesh)
        divide_cell(mesh_test, fid, c_try, u_try)
        face_id, c, u, c1, c2 = fid, c_try, u_try, c1_try, c2_try
        v0s, v1s = v0s_try, v1s_try
        break
    except Exception as e:
        print(f"  face {fid} skipped: {e}")
        continue

if face_id is None:
    raise RuntimeError("No divisible cell found.")

print(f"Selected cell: {face_id}")
print(f"Center: ({c[0]:.4f}, {c[1]:.4f})")
print(f"Spindle axis u: ({u[0]:.4f}, {u[1]:.4f})")


In [ ]:
from visualization import _plot_spindle_pair
# Two views: the classic full-tissue plot, and a zoom on the dividing cell
# showing the two centrosomes (C1, C2), the geometric centroid, the mitotic
# spindle axis, and the angle φ.
_plot_spindle_pair(mesh, face_id, c, u, c1, c2, v0s, v1s)
plt.show()

## Cell division and relaxation

Cut the selected cell along the spindle axis, then relax the tissue with the bilevel
inner optimization. The mesh is shown **just after division** (left) and **after
relaxation** (right), side by side.

In [ ]:
mesh_post, _ = divide_cell(mesh, face_id, c, u)

print(f"Before: {mesh.nb_faces} cells")
print(f"After: {mesh_post.nb_faces} cells")

In [ ]:
mesh_opt = BoundedMesh.copy_mesh(mesh_post)
# Fixed per-cell target area: every cell relaxes towards 0.6 (no total-area target),
# so the tissue's total area grows as cells divide.
target_area = 0.6
mesh_opt.faces_params = jnp.full(mesh_opt.nb_faces, target_area)
print(f"Per-cell target area: {target_area:.4f} ({mesh_opt.nb_faces} cells "
      f"→ total target {target_area * mesh_opt.nb_faces:.4f})")

opt = BoundedBilevelOptimizer()
opt.loss_function_inner = energy_adaptive
opt.inner_solver        = optax.sgd(learning_rate=1e-3)
opt.update_T1           = True
opt._update_T1_func     = update_T1_bounded
opt.min_dist_T1         = 0.01
opt.max_nb_iterations   = 1000
opt.tolerance           = 1e-6
opt.patience            = 50

print(f"Before: {mesh_opt.nb_faces} cells, {mesh_opt.nb_edges} edges")
loss_hist = opt.inner_optimization(mesh_opt)
print(f"After: {mesh_opt.nb_faces} cells, {mesh_opt.nb_edges} edges")
print(f"{len(loss_hist)} iterations, loss = {float(loss_hist[-1]):.4f}")

In [ ]:
from visualization import _plot_div_relax_pair
# Left: mesh just after division.  Right: mesh after relaxation (inner optimization).
# Colour scales are taken from `mesh_initial` (the raw initial mesh, same scale as the
# "before relaxation" panel) so the areas/tensions are comparable across the notebook.
_plot_div_relax_pair(mesh_post, mesh_opt, division_idx=1, mesh_ref=mesh_initial)
plt.show()

## 5 consecutive divisions


In [ ]:
import importlib
import spindle_analysis; importlib.reload(spindle_analysis)
from spindle_analysis import run_n_divisions

mesh_newton = BoundedMesh.copy_mesh(mesh)

mesh_final, division_snapshots = run_n_divisions(
    mesh_newton, n_divisions=5,
    cell_rng=np.random.default_rng(0),
)
print(f"Final tissue: {mesh_final.nb_faces} cells, {mesh_final.nb_edges} edges")


In [ ]:
from visualization import (_draw_mesh_on_ax, _add_cell_labels,
                           AREA_SCALE, TENSION_SCALE)
import matplotlib
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

sm_tension = ScalarMappable(Normalize(*TENSION_SCALE), matplotlib.colormaps["coolwarm"])
sm_area = ScalarMappable(Normalize(*AREA_SCALE), matplotlib.colormaps["cividis"])


def _row_colorbars(fig, row_axes):
    """Add a tension + area colorbar pair to the right of one grid row. Labels are
    put on the LEFT of each bar so they never collide with the neighbouring bar."""
    cbt = fig.colorbar(sm_tension, ax=row_axes, fraction=0.035, pad=0.03, aspect=15)
    cba = fig.colorbar(sm_area, ax=row_axes, fraction=0.035, pad=0.07, aspect=15)
    for cb, lbl in ((cbt, "tension"), (cba, "Area of cell")):
        cb.ax.yaxis.set_label_position("left")
        cb.set_label(lbl)


fig, axes = plt.subplots(5, 2, figsize=(16, 25))
fig.suptitle("5 divisions  |  left: just after division  ·  right: after relaxation",
             fontsize=13)

for div_idx, (snap_post, snap_opt) in enumerate(division_snapshots, start=1):
    row = div_idx - 1
    for col, (snap, label) in enumerate([
            (snap_post, f"Division {div_idx} — just after ({snap_post.nb_faces} cells)"),
            (snap_opt,  f"Division {div_idx} — after relaxation ({snap_opt.nb_faces} cells)"),
    ]):
        ax = axes[row, col]
        ta = float(jnp.sum(snap.get_area(jnp.arange(snap.nb_faces))))
        # Common scales (tension = sigmoid(he_params)+1 ∈ [1,2]; area = initial mesh).
        _draw_mesh_on_ax(snap, ax, AREA_SCALE, TENSION_SCALE)
        _add_cell_labels(ax, snap)
        ax.set_title(f"{label} — area: {ta:.3f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    _row_colorbars(fig, axes[row, :])

plt.show()

## Area decomposition of cells 21, 23, 10 (division 4, after relaxation)

The bounded-mesh area (`get_area_bounded`, Green's theorem) is the sum of two terms per
edge: a **shoelace** term over the chord vertices, and, for **boundary edges only**, the
**circular-segment** area between the chord and the arc (`chordlen² · f(angle)`). Below we
recompute that decomposition for cells 21, 23, 10 and draw each cell zoomed, showing the
edges actually used: **blue** = internal edges (straight chord, shoelace only), **red** =
boundary arc, **gray dashed** = the arc's chord.

In [ ]:
from vertax.meshes.plot import _get_arc_with_n_points
from vertax.geo import _compute_contributions

# Division 4, after relaxation.
_m_area = division_snapshots[3][1]

# Same tables get_area_bounded uses internally: vertTable prefixed with the two
# reserved rows [0,0] and [1,1], and per-half-edge angles. (Names are underscored
# to avoid clobbering globals used by other cells.)
_vtab = jnp.vstack([jnp.array([[0.0, 0.0], [1.0, 1.0]]), jnp.asarray(_m_area.vertices)])
_atab = jnp.repeat(jnp.asarray(_m_area.angles), 2)
_htab = jnp.asarray(_m_area.edges)
_E = np.array(_m_area.edges)
_Vp = np.array(_m_area.vertices)


def _he_chain(face):
    """Half-edges around a face, following the `next` pointer (col 1)."""
    start = int(_m_area.faces[face][0]); he = start; out = []
    for _ in range(60):
        out.append(he); he = int(_E[he][1])
        if he == start:
            break
    return out


def _decompose(face):
    """Shoelace + circular-segment split using vertax's own _compute_contributions,
    so the numbers match get_area_bounded exactly."""
    sh_t = 0.0; sg_t = 0.0
    for he in _he_chain(face):
        sh, sg = _compute_contributions(jnp.int32(he), _vtab, _atab, _htab)
        sh_t += float(sh); sg_t += float(sg)
    return 0.5 * abs(sh_t + sg_t), 0.5 * sh_t, 0.5 * sg_t


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, f in zip(axes, (21, 23, 10)):
    area, lace_a, seg_a = _decompose(f)
    api = float(_m_area.get_area(jnp.array([f]))[0])
    print(f"cell {f}: area={api:.4f}  | shoelace={lace_a:.4f}  circular-segment={seg_a:.4f}")
    # Draw the edges actually used: internal chords (blue), boundary arcs (red) + chord (gray).
    for he in _he_chain(f):
        internal = int(_E[he][3]) != 0
        vs = int(_E[he][3] + _E[he][5] - 2); vt_id = int(_E[he][4] + _E[he][6] - 3)
        if not (0 <= vs < len(_Vp) and 0 <= vt_id < len(_Vp)):
            continue
        p0, p1 = _Vp[vs], _Vp[vt_id]
        if internal:
            ax.plot([p0[0], p1[0]], [p0[1], p1[1]], "-", color="tab:blue", lw=2)
        else:
            ang = float(_m_area.angles[he // 2])
            arc = np.array(_get_arc_with_n_points(ang, p0, p1))
            pts = np.vstack([p0, arc, p1])
            ax.plot(pts[:, 0], pts[:, 1], "-", color="tab:red", lw=2)
            ax.plot([p0[0], p1[0]], [p0[1], p1[1]], "--", color="gray", lw=1)
        ax.scatter([p0[0]], [p0[1]], color="k", s=25, zorder=5)
    ax.set_aspect("equal")
    ax.set_title(f"cell {f}\narea={api:.4f}  (shoelace {lace_a:.3f} + arc {seg_a:.3f})", fontsize=10)

fig.suptitle("Area decomposition — blue: internal chord · red: boundary arc · gray dashed: arc chord",
             fontsize=12)
plt.show()

## 5 consecutive divisions — Théry et al. 2007 orientation model

The `'pdf'` / `'pdf_sample'` methods implement the spindle-orientation model of
**Théry et al. (2007, Nature 447:493)**, following the exact equations of their
Supplementary Information (S1–S15). The only adaptation to a tissue context is the
cortical force cue: where Théry use the density of *retraction fibres* of an
adhesive micropattern, we use the **junctional tension** $\sigma$ of the vertex
model — a legitimate instance of their general assumption that *any* source of
cortical heterogeneity activates the force generators (their Suppl. Discussion).

### Model (Théry et al., Suppl. Eqs. S1–S15)

The cell cortex is sampled at cortical angles $\psi$; the boundary point is

$$\mathbf{R}(\psi) = R\begin{pmatrix}\cos\psi\\ \sin\psi\end{pmatrix}, \qquad \text{(S1)}$$

(on the real cell boundary here, rather than an exact circle). The spindle has two
poles at $\mathbf{c}\pm a\,\mathbf{u}(\varphi)$ with
$\mathbf{u}(\varphi)=(\cos\varphi,\sin\varphi)$. Astral microtubules from each pole
reach the cortex; the one hitting $\psi$ points along the unit vector

$$\hat{\mathbf{m}}(\psi,\varphi) = \begin{pmatrix}\cos\gamma(\psi,\varphi)\\
\sin\gamma(\psi,\varphi)\end{pmatrix}, \qquad \gamma = \alpha(\theta)+\varphi,
\quad \theta=\psi-\varphi, \qquad \text{(S2)}$$

i.e. the direction from the pole to $\mathbf{R}(\psi)$. The **cortical force per
angular element** is

$$\mathbf{f}(\psi,\varphi) = F(\psi)\,\rho_{MT}(\psi-\varphi)\,\hat{\mathbf{m}}(\psi,\varphi),
\qquad \text{(S3)}$$

with the **angular microtubule density**

$$\rho_{MT}(\theta) = \frac{N_{MT}}{2\pi}\,\frac{\mathrm{d}\alpha}{\mathrm{d}\theta},
\qquad \theta=\psi-\varphi, \qquad \text{(S4)}$$

where $\alpha$ is the emission angle at the pole; each pole emits only towards the
half of the cortex it faces. The force generators exert a **net torque**

$$\tau_z(\varphi) = \int_{-\pi}^{\pi}\!\big(R_x f_y - R_y f_x\big)\,\mathrm{d}\psi,
\qquad \text{(S5)}$$

summed over the two poles, which defines the **orientational potential**

$$W(\varphi) = -\!\int^{\varphi}\!\tau_z(\varphi')\,\mathrm{d}\varphi'. \qquad \text{(S6)}$$

Stable spindle orientations are the minima of $W$. A Langevin equation for
$\varphi$ with friction $\eta$ and white noise (Eqs. S10–S11) yields, at steady
state, the **angular distribution**

$$P(\varphi) = N\,\exp\!\Big(-\frac{W(\varphi)}{d}\Big), \qquad \text{(S12/S14)}$$

where $d$ is a single dimensionless noise parameter (Eq. S15) combining the noise
strength, friction, and force-generator coupling. Here $F(\psi)=\sigma(\psi)$ (the
junctional tension reached at $\psi$).

- **`'pdf'`** takes the mode of $P(\varphi)$ — the most stable orientation
  (deterministic, independent of $d$).
- **`'pdf_sample'`** draws $\varphi\sim P(\varphi)$ — the spread reproduces the
  experimental cell-to-cell variability; larger $d$ widens it.

In [ ]:
import importlib
import spindle_analysis; importlib.reload(spindle_analysis)
from spindle_analysis import run_n_divisions

# Start from the relaxed mesh so the PDF divisions begin from a
# mechanically equilibrated tissue, consistent with all other sections.
mesh_pdf = BoundedMesh.copy_mesh(mesh)

mesh_final_pdf, division_snapshots_pdf = run_n_divisions(
    mesh_pdf, n_divisions=5, spindle_method='pdf', spindle_pdf_C=0.1,
    cell_rng=np.random.default_rng(0),
)
print(f"Final tissue (pdf): {mesh_final_pdf.nb_faces} cells, {mesh_final_pdf.nb_edges} edges")


In [ ]:
from visualization import _draw_mesh_on_ax, _add_cell_labels, AREA_SCALE, TENSION_SCALE
import matplotlib
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

sm_tension = ScalarMappable(Normalize(*TENSION_SCALE), matplotlib.colormaps["coolwarm"])
sm_area = ScalarMappable(Normalize(*AREA_SCALE), matplotlib.colormaps["cividis"])


def _row_colorbars(fig, row_axes):
    cbt = fig.colorbar(sm_tension, ax=row_axes, fraction=0.035, pad=0.03, aspect=15)
    cba = fig.colorbar(sm_area, ax=row_axes, fraction=0.035, pad=0.07, aspect=15)
    for cb, lbl in ((cbt, "tension"), (cba, "Area of cell")):
        cb.ax.yaxis.set_label_position("left")
        cb.set_label(lbl)


fig, axes = plt.subplots(5, 2, figsize=(16, 25))
fig.suptitle("5 divisions (pdf mode)  |  left: just after division  ·  right: after relaxation",
             fontsize=13)

for div_idx, (snap_post, snap_opt) in enumerate(division_snapshots_pdf, start=1):
    row = div_idx - 1
    for col, (snap, label) in enumerate([
            (snap_post, f"Division {div_idx} — just after ({snap_post.nb_faces} cells)"),
            (snap_opt,  f"Division {div_idx} — after relaxation ({snap_opt.nb_faces} cells)"),
    ]):
        ax = axes[row, col]
        ta = float(jnp.sum(snap.get_area(jnp.arange(snap.nb_faces))))
        _draw_mesh_on_ax(snap, ax, AREA_SCALE, TENSION_SCALE)
        _add_cell_labels(ax, snap)
        ax.set_title(f"{label} — area: {ta:.3f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    _row_colorbars(fig, axes[row, :])

plt.show()

## 5 consecutive divisions — PDF method (stochastic sample)

The `'pdf_sample'` method uses the same density $p(\alpha)$ defined above, but **randomly draws** the division axis from that distribution rather than taking its mode.

This models the experimental observation that identical cells do not always divide along the same axis: the parameter $C$ quantifies the deviation from the deterministic equilibrium — larger $C$ makes atypical orientations more frequent.

In [ ]:
import importlib
import spindle_analysis; importlib.reload(spindle_analysis)
from spindle_analysis import run_n_divisions

mesh_sample = BoundedMesh.copy_mesh(mesh)

mesh_final_sample, division_snapshots_sample = run_n_divisions(
    mesh_sample, n_divisions=5, spindle_method='pdf_sample', spindle_pdf_C=0.3,
    spindle_rng=np.random.default_rng(0),
    cell_rng=np.random.default_rng(0),
)
print(f"Final tissue (pdf_sample): {mesh_final_sample.nb_faces} cells, {mesh_final_sample.nb_edges} edges")


In [ ]:
from visualization import _draw_mesh_on_ax, _add_cell_labels, AREA_SCALE, TENSION_SCALE
import matplotlib
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

sm_tension = ScalarMappable(Normalize(*TENSION_SCALE), matplotlib.colormaps["coolwarm"])
sm_area = ScalarMappable(Normalize(*AREA_SCALE), matplotlib.colormaps["cividis"])


def _row_colorbars(fig, row_axes):
    cbt = fig.colorbar(sm_tension, ax=row_axes, fraction=0.035, pad=0.03, aspect=15)
    cba = fig.colorbar(sm_area, ax=row_axes, fraction=0.035, pad=0.07, aspect=15)
    for cb, lbl in ((cbt, "tension"), (cba, "Area of cell")):
        cb.ax.yaxis.set_label_position("left")
        cb.set_label(lbl)


fig, axes = plt.subplots(5, 2, figsize=(16, 25))
fig.suptitle("5 divisions (pdf_sample, C=0.3)  |  left: just after division  ·  right: after relaxation",
             fontsize=13)

for div_idx, (snap_post, snap_opt) in enumerate(division_snapshots_sample, start=1):
    row = div_idx - 1
    for col, (snap, label) in enumerate([
            (snap_post, f"Division {div_idx} — just after ({snap_post.nb_faces} cells)"),
            (snap_opt,  f"Division {div_idx} — after relaxation ({snap_opt.nb_faces} cells)"),
    ]):
        ax = axes[row, col]
        ta = float(jnp.sum(snap.get_area(jnp.arange(snap.nb_faces))))
        _draw_mesh_on_ax(snap, ax, AREA_SCALE, TENSION_SCALE)
        _add_cell_labels(ax, snap)
        ax.set_title(f"{label} — area: {ta:.3f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    _row_colorbars(fig, axes[row, :])

plt.show()

## Cell polarity bias on spindle orientation

### Biological motivation

In epithelial tissues, the mitotic spindle aligns not only with cell geometry (Hertwig's rule) but also with the planar polarity of the tissue. Planar cell polarity (PCP) is encoded in the asymmetric distribution of cortical tension: cells are pulled more strongly along certain directions by their junctions, creating a mechanical polarity axis. Experimental work by Théry et al. and Minc et al. shows that both geometry and tension polarity contribute to division axis selection — neither fully controls it.

### Polarity axis from the tension field

After each mechanical relaxation, the **polarity axis** $\varphi_f$ of cell $f$ is extracted from the tension-weighted edge-direction tensor:

$$\mathbf{T}_f = \sum_{e \in f} \sigma_e \,\hat{\mathbf{e}}_e \otimes \hat{\mathbf{e}}_e$$

where $\hat{\mathbf{e}}_e = (\mathbf{v}_1 - \mathbf{v}_0)/|\mathbf{v}_1 - \mathbf{v}_0|$ is the unit direction of junction $e$ and $\sigma_e = \text{sigmoid}(\theta_e) + 1$ is its tension (from `he_params`).

$\varphi_f$ is the angle of the **principal eigenvector** of $\mathbf{T}_f$ — the direction along which the cell is most strongly pulled.

### Polarity-biased microtubule density

The polarity axis biases spindle orientation by modulating the microtubule emission density:

$$\Lambda(\theta) = \sigma_j \cdot \bigl(1 + p \cos^2(\theta - \varphi_f)\bigr)$$

| Symbol | Meaning |
|--------|---------|
| $\theta$ | Emission direction of a microtubule ray |
| $\sigma_j$ | Tension of the membrane segment reached at angle $\theta$ |
| $\varphi_f$ | Polarity axis (principal tension eigenvector) |
| $p$ | `polarity_strength` $\in [0, 1]$ |

At $p = 0$ the model is identical to the geometry-only case (standard Newton / PDF methods). At $p = 1$, the microtubule density is doubled along $\varphi_f$ without suppressing other directions — the spindle remains free to align with cell geometry when it conflicts with polarity, matching the experimental observation that both cues contribute.

Polarity is recomputed from the tension field at each division step: no explicit tracking or inheritance is needed, since the mechanical state after relaxation encodes the tissue's history.

In [ ]:
import importlib
import spindle_analysis; importlib.reload(spindle_analysis)
from spindle_analysis import run_n_divisions

# cell_rng is seeded identically so the same cells are chosen as in the
# Newton run (division_snapshots) — only the spindle orientation changes.
mesh_pol = BoundedMesh.copy_mesh(mesh)

print("Running 5 divisions WITH polarity bias (p=1.0)...")
mesh_final_pol, snaps_pol = run_n_divisions(
    mesh_pol, n_divisions=5, polarity_strength=1.0,
    cell_rng=np.random.default_rng(0),
)

print(f"With polarity: {mesh_final_pol.nb_faces} cells")


In [ ]:
from visualization import _draw_mesh_on_ax, _add_cell_labels, AREA_SCALE, TENSION_SCALE
import matplotlib
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

sm_tension = ScalarMappable(Normalize(*TENSION_SCALE), matplotlib.colormaps["coolwarm"])
sm_area    = ScalarMappable(Normalize(*AREA_SCALE),    matplotlib.colormaps["cividis"])


def _row_colorbars(fig, row_axes):
    cbt = fig.colorbar(sm_tension, ax=row_axes, fraction=0.035, pad=0.03, aspect=15)
    cba = fig.colorbar(sm_area,    ax=row_axes, fraction=0.035, pad=0.07, aspect=15)
    for cb, lbl in ((cbt, "tension"), (cba, "Area of cell")):
        cb.ax.yaxis.set_label_position("left")
        cb.set_label(lbl)


fig, axes = plt.subplots(5, 2, figsize=(16, 25))
fig.suptitle("5 divisions — PCP (polarity p=1.0)  |  left: just after division  ·  right: after relaxation",
             fontsize=13)

for div_idx, (snap_post, snap_opt) in enumerate(snaps_pol, start=1):
    row = div_idx - 1
    for col, (snap, label) in enumerate([
        (snap_post, f"Division {div_idx} — just after ({snap_post.nb_faces} cells)"),
        (snap_opt,  f"Division {div_idx} — after relaxation ({snap_opt.nb_faces} cells)"),
    ]):
        ax = axes[row, col]
        ta = float(jnp.sum(snap.get_area(jnp.arange(snap.nb_faces))))
        _draw_mesh_on_ax(snap, ax, AREA_SCALE, TENSION_SCALE)
        _add_cell_labels(ax, snap)
        ax.set_title(f"{label} — area: {ta:.3f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    _row_colorbars(fig, axes[row, :])

plt.show()


## Edge polarity scores biasing the spindle orientation

Here we start from the **base Newton spindle** computed in the first section (orientation
$\varphi_0$, axis $\mathbf{u}$) and **bias it probabilistically** using a per-edge
**polarity score** $w_e$ (random, fixed seed).

Each boundary segment $e$ has a direction $\theta_e$ and a score $w_e$. We build a density
over candidate axis orientations $\alpha \in [-\tfrac{\pi}{2}, \tfrac{\pi}{2}]$ that keeps
the Newton spindle as an **anchor** and lets the edge scores **pull** the orientation
towards high-score directions:

$$p(\alpha) \;\propto\; \underbrace{\exp\!\big(\kappa\,\cos 2(\alpha-\varphi_0)\big)}_{\text{anchor on }\varphi_0}
\;\cdot\;\underbrace{\exp\!\Big(\beta \sum_e w_e\,\ell_e\,\cos^2(\alpha-\theta_e)\Big)}_{\text{edge-score bias}}$$

- the **anchor** (a von Mises-like term on the *axis*, hence $\cos 2(\cdot)$) keeps $\alpha$
  near the mechanically-balanced Newton orientation; $\kappa$ sets how tightly;
- the **bias** raises the probability of orientations aligned with high-score edges
  (weighted by segment length $\ell_e$); $\beta$ sets its strength;
- the final orientation $\varphi$ is then **drawn stochastically** from $p(\alpha)$
  (fixed seed → reproducible).

The plot shows the base Newton spindle ($\varphi_0$), the biased draw ($\varphi$), the
density $p(\alpha)$, and the per-edge scores on the cell.

In [ ]:
from mesh_utils import get_face_polygon
from vertax.meshes.plot import _get_arc_with_n_points
import matplotlib

# --- Per-EDGE polarity scores (one score per edge, fixed seed) --------------------
# get_cell_segments subdivides curved boundary arcs into several straight
# sub-segments; if we scored per sub-segment a single arc would get several
# scores. So we walk the face's half-edges ourselves, assign ONE random score per
# edge (he // 2), and propagate it unchanged to every sub-segment of that edge.
_E = np.array(mesh.edges); _V = np.array(mesh.vertices); _ANG = np.array(mesh.angles)
_score_rng = np.random.default_rng(0)
_edge_score = {}  # edge id -> score, drawn once per edge

_seg_v0, _seg_v1, _seg_w = [], [], []
_start = int(mesh.faces[face_id][0]); _he = _start; _first = True
while _first or _he != _start:
    _first = False
    _eid = _he // 2
    if _eid not in _edge_score:
        _edge_score[_eid] = float(_score_rng.uniform(0.0, 1.0))
    wv = _edge_score[_eid]
    src = int(_E[_he, 3] + _E[_he, 5] - 2); tgt = int(_E[_he, 4] + _E[_he, 6] - 3)
    if src >= 0 and tgt >= 0:
        p0, p1 = _V[src], _V[tgt]
        ang = float(_ANG[_eid])
        is_surface = int(_E[_he, 3]) == 0
        if is_surface and ang > 0.01:
            pts = [p0] + _get_arc_with_n_points(ang, p0, p1) + [p1]  # one arc -> sub-segments
            for a, b in zip(pts[:-1], pts[1:]):
                _seg_v0.append(a); _seg_v1.append(b); _seg_w.append(wv)  # same score on all
        else:
            _seg_v0.append(p0); _seg_v1.append(p1); _seg_w.append(wv)
    _he = int(_E[_he, 1])

_seg_v0 = np.array(_seg_v0); _seg_v1 = np.array(_seg_v1); _w = np.array(_seg_w)
_dirs = _seg_v1 - _seg_v0
_theta = np.arctan2(_dirs[:, 1], _dirs[:, 0])          # segment direction
_length = np.hypot(_dirs[:, 0], _dirs[:, 1])           # segment length ℓ_e

# --- Base Newton orientation φ0 (folded onto the axis [-90°, 90°]) ----------------
phi0 = float(np.arctan2(u[1], u[0]))
phi0 = (phi0 + np.pi / 2) % np.pi - np.pi / 2

# --- Biased density p(α) = anchor(φ0) · edge-score bias ---------------------------
KAPPA = 4.0   # anchor strength on the Newton orientation
BETA = 4.0    # edge-score bias strength
alphas = np.linspace(-np.pi / 2, np.pi / 2, 361)
anchor = np.exp(KAPPA * np.cos(2 * (alphas - phi0)))
bias = np.exp(BETA * np.array([np.sum(_w * _length * np.cos(a - _theta) ** 2) for a in alphas]))
p_alpha = anchor * bias
p_alpha /= np.trapz(p_alpha, alphas)

# --- Stochastic draw of the biased orientation φ (fixed seed) ---------------------
_draw_rng = np.random.default_rng(1)
phi = float(_draw_rng.choice(alphas, p=p_alpha / p_alpha.sum()))
print(f"Newton φ0 = {np.degrees(phi0):+.1f}°   biased draw φ = {np.degrees(phi):+.1f}°   "
      f"(mode {np.degrees(alphas[np.argmax(p_alpha)]):+.1f}°)")

# --- Plot: edge scores + spindle axes | density p(α) -----------------------------
poly = get_face_polygon(face_id, mesh)
ctr = poly.mean(axis=0)
half = 0.5 * float(np.max(np.linalg.norm(poly - ctr, axis=1)))
cmap = matplotlib.colormaps["viridis"]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1 — cell, per-edge scores (colour), base vs biased spindle axis
for (a, b, wv) in zip(_seg_v0, _seg_v1, _w):
    ax0.plot([a[0], b[0]], [a[1], b[1]], color=cmap(wv), lw=3, solid_capstyle="round")
for ang, col, lbl in [(phi0, "#1a7abf", f"Newton φ0 = {np.degrees(phi0):+.0f}°"),
                      (phi, "#e63946", f"biased φ = {np.degrees(phi):+.0f}°")]:
    d = np.array([np.cos(ang), np.sin(ang)])
    ax0.plot([ctr[0] - half * d[0], ctr[0] + half * d[0]],
             [ctr[1] - half * d[1], ctr[1] + half * d[1]],
             color=col, lw=2.5, label=lbl)
ax0.scatter(*ctr, color="k", s=20, zorder=5)
ax0.set_aspect("equal"); ax0.legend(loc="upper right", fontsize=9)
ax0.set_title(f"Cell {face_id} — per-edge polarity scores + spindle axis")
fig.colorbar(matplotlib.cm.ScalarMappable(matplotlib.colors.Normalize(0, 1), cmap),
             ax=ax0, fraction=0.046, pad=0.04, label="edge polarity score $w_e$")

# Panel 2 — biased density p(α)
ax1.plot(np.degrees(alphas), p_alpha, color="k", lw=1.5)
ax1.axvline(np.degrees(phi0), color="#1a7abf", lw=2, label="Newton φ0")
ax1.axvline(np.degrees(phi), color="#e63946", lw=2, ls="--", label="biased draw φ")
ax1.set_xlabel("orientation α (deg)"); ax1.set_ylabel("p(α)")
ax1.set_title("Biased orientation density  p(α) ∝ anchor(φ0) · edge-score bias")
ax1.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
import importlib
import spindle_analysis; importlib.reload(spindle_analysis)
from spindle_analysis import run_n_divisions

# 5 divisions where the Newton spindle axis is biased at each step by per-edge
# polarity scores (same anchor/bias strengths as the single-cell demo above).
# Same cell_rng(0) as the other methods so the same cells are picked each step.
mesh_edge = BoundedMesh.copy_mesh(mesh)

mesh_final_edge, snaps_edge = run_n_divisions(
    mesh_edge, n_divisions=5,
    cell_rng=np.random.default_rng(0),
    edge_bias=True, edge_bias_kappa=KAPPA, edge_bias_beta=BETA,
    edge_bias_rng=np.random.default_rng(0),
)
print(f"Final tissue (edge-bias): {mesh_final_edge.nb_faces} cells, {mesh_final_edge.nb_edges} edges")

In [ ]:
from visualization import _draw_mesh_on_ax, _add_cell_labels, AREA_SCALE, TENSION_SCALE
import matplotlib
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

sm_tension = ScalarMappable(Normalize(*TENSION_SCALE), matplotlib.colormaps["coolwarm"])
sm_area    = ScalarMappable(Normalize(*AREA_SCALE),    matplotlib.colormaps["cividis"])


def _row_colorbars(fig, row_axes):
    cbt = fig.colorbar(sm_tension, ax=row_axes, fraction=0.035, pad=0.03, aspect=15)
    cba = fig.colorbar(sm_area,    ax=row_axes, fraction=0.035, pad=0.07, aspect=15)
    for cb, lbl in ((cbt, "tension"), (cba, "Area of cell")):
        cb.ax.yaxis.set_label_position("left")
        cb.set_label(lbl)


fig, axes = plt.subplots(5, 2, figsize=(16, 25))
fig.suptitle("5 divisions — edge-score bias  |  left: just after division  ·  right: after relaxation",
             fontsize=13)

for div_idx, (snap_post, snap_opt) in enumerate(snaps_edge, start=1):
    row = div_idx - 1
    for col, (snap, label) in enumerate([
        (snap_post, f"Division {div_idx} — just after ({snap_post.nb_faces} cells)"),
        (snap_opt,  f"Division {div_idx} — after relaxation ({snap_opt.nb_faces} cells)"),
    ]):
        ax = axes[row, col]
        ta = float(jnp.sum(snap.get_area(jnp.arange(snap.nb_faces))))
        _draw_mesh_on_ax(snap, ax, AREA_SCALE, TENSION_SCALE)
        _add_cell_labels(ax, snap)
        ax.set_title(f"{label} — area: {ta:.3f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    _row_colorbars(fig, axes[row, :])

plt.show()

## Comparison of division methods — post-division (pre-relaxation)

Each row is one division step (same cell chosen at each step across all methods,
`cell_rng=rng(0)`). The columns show the tissue **just after the cut**, before
any relaxation, for each spindle model:

| Column | Method |
|--------|--------|
| 1 | Newton (deterministic force/torque balance) |
| 2 | PDF mode (most probable orientation) |
| 3 | PDF sample (stochastic draw from density) |
| 4 | PCP — polarity-biased Newton (p = 1.0) |
| 5 | Edge-score bias — Newton axis biased by per-edge polarity scores |


In [ ]:
from visualization import _draw_mesh_on_ax, _add_cell_labels, AREA_SCALE, TENSION_SCALE
import matplotlib
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

sm_tension = ScalarMappable(Normalize(*TENSION_SCALE), matplotlib.colormaps["coolwarm"])
sm_area    = ScalarMappable(Normalize(*AREA_SCALE),    matplotlib.colormaps["cividis"])


def _row_colorbars5(fig, row_axes):
    cbt = fig.colorbar(sm_tension, ax=row_axes, fraction=0.012, pad=0.02, aspect=15)
    cba = fig.colorbar(sm_area,    ax=row_axes, fraction=0.012, pad=0.05, aspect=15)
    for cb, lbl in ((cbt, "tension"), (cba, "Area of cell")):
        cb.ax.yaxis.set_label_position("left")
        cb.set_label(lbl)


methods = [
    (division_snapshots,        "Newton"),
    (division_snapshots_pdf,    "PDF mode"),
    (division_snapshots_sample, "PDF sample"),
    (snaps_pol,                 "PCP p=1.0"),
    (snaps_edge,                "Edge-bias"),
]

n_divs = min(len(s) for s, _ in methods)
ncol = len(methods)
fig, axes = plt.subplots(n_divs, ncol, figsize=(7 * ncol, 5 * n_divs))
fig.suptitle(
    "Division methods — post-division, pre-relaxation  "
    "|  cols: Newton · PDF mode · PDF sample · PCP (p=1.0) · Edge-bias",
    fontsize=12,
)

for row, div_idx in enumerate(range(n_divs)):
    for col, (snaps, method_name) in enumerate(methods):
        snap_post, _ = snaps[div_idx]
        ax = axes[row, col]
        ta = float(jnp.sum(snap_post.get_area(jnp.arange(snap_post.nb_faces))))
        _draw_mesh_on_ax(snap_post, ax, AREA_SCALE, TENSION_SCALE)
        _add_cell_labels(ax, snap_post)
        ax.set_title(f"Div {div_idx + 1} — {method_name} ({snap_post.nb_faces}c) — area: {ta:.3f}",
                     fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    _row_colorbars5(fig, axes[row, :])

plt.show()


## Inner-solver benchmark — which solver relaxes fastest over 25 divisions

The mechanical relaxation after each division is an inner optimization
(`BoundedBilevelOptimizer.inner_optimization`) that minimises `energy_adaptive`.
Here we compare four optax-compatible inner solvers on the **same** division
sequence — a **4-cell** bounded mesh subjected to **25 consecutive divisions**
(ending at 29 cells) — and measure the **wall-clock time** of each full run.

| Solver | Construction | Kind |
|--------|--------------|------|
| **SGD**    | `optax.sgd(learning_rate=1e-3)` | plain gradient descent |
| **Adam**   | `optax.adam(learning_rate=1e-3)` | adaptive first-order |
| **CG**     | `vertax.nonlinear_cg(restart_every=10)` | nonlinear conjugate gradient (Polak–Ribière + zoom line search) |
| **L-BFGS** | `optax.lbfgs()` | limited-memory quasi-Newton (zoom line search) |

Everything else is held fixed: same starting mesh, same `cell_rng(0)` (so each
solver picks the **same** cells and division axes at every step), same
`energy_adaptive`, same T1 settings, same `max_opt_iter`. Only `inner_solver`
changes, via the `inner_solver=` argument of `run_n_divisions`.

Each solver runs in its **own cell** so its timing is recorded independently;
the final cells plot the **initial** and **final** mesh per solver and summarise
the timings.

> **Note** — `optax.lbfgs` requires **optax ≥ 0.2.5** here: in 0.2.4 its zoom
> line-search state mixes `int32`/`int64`, which breaks the early-stopping
> `lax.cond` inside `inner_optimization` under `JAX_ENABLE_X64`. The project pins
> `optax==0.2.5`.

In [ ]:
# --- Self-contained setup: works even if run right after a kernel restart -------
# jax 0.4.35 crashes auto-detecting the CUDA path (cuda_nvcc.__file__ is None), so
# CUDA_ROOT must be set BEFORE the first `import jax`. The notebook's top imports
# cell already does this; we repeat it here so this benchmark section is runnable
# on its own without hitting the misleading "partially initialized jax" error.
import os, sys, math, sysconfig
sys.path.insert(0, '/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('JAX_ENABLE_X64', 'true')
_nvcc = os.path.join(sysconfig.get_paths()['purelib'], 'nvidia', 'cuda_nvcc')
if os.path.isdir(_nvcc):
    os.environ.setdefault('CUDA_ROOT', _nvcc)

import time
import importlib
import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

from vertax import BoundedMesh, nonlinear_cg
import energy_adaptive as _ea_mod; importlib.reload(_ea_mod)
from energy_adaptive import energy_adaptive
import spindle_analysis; importlib.reload(spindle_analysis)
from spindle_analysis import run_n_divisions

# Benchmark settings: a small 4-cell tissue (minimum for a Voronoi mesh) taken
# through 25 consecutive divisions -> 29 cells. Each solver gets its OWN iteration
# budget and early-stopping criterion, passed explicitly in its cell below.
BENCH_N_CELLS    = 4
BENCH_N_DIV      = 25
BENCH_TARGET_AREA = 0.6
# Default early-stopping criterion (per solver cell may override).
BENCH_TOLERANCE  = 1e-6
BENCH_PATIENCE   = 50


def mesh_metrics(m):
    """Energy decomposition + area/tension stats of a relaxed mesh, computed with
    the SAME energy that is minimised (energy_adaptive). Returns a dict with the
    global energy, its area vs tension parts, and mean area / mean tension."""
    vt = jnp.asarray(m.vertices); at = jnp.asarray(m.angles)
    he = jnp.asarray(m.edges);    fc = jnp.asarray(m.faces)
    vp = jnp.asarray(m.vertices_params)
    ep = jnp.asarray(m.edges_params); fp = jnp.asarray(m.faces_params)

    E_total = float(energy_adaptive(vt, at, he, fc, None, None, None, vp, ep, fp))

    # Area part: K_AREAS * sum (area - target)^2 (target = fp[0]); tension part is
    # the remainder of the total (inner + surface edge energy).
    areas = m.get_area(jnp.arange(m.nb_faces))
    E_area = float(_ea_mod.K_AREAS * jnp.sum((areas - fp[0]) ** 2))
    E_tension = E_total - E_area
    tensions = jax.nn.sigmoid(ep) + 1
    return dict(E=E_total, E_area=E_area, E_tension=E_tension,
                mean_area=float(jnp.mean(areas)), total_area=float(jnp.sum(areas)),
                mean_tension=float(jnp.mean(tensions)))


def make_bench_mesh():
    """Fresh 4-cell bounded mesh with the same tension/area conventions used
    throughout this notebook (so every solver starts from an identical mesh)."""
    w = math.sqrt(BENCH_N_CELLS)
    m = BoundedMesh.from_random_seeds(nb_seeds=BENCH_N_CELLS, width=w, height=w, random_key=2)
    m.vertices_params = jnp.asarray([0.0 for _ in range(m.nb_vertices)])
    key = jax.random.PRNGKey(3)
    m.edges_params = 1 + jax.nn.sigmoid(jax.random.uniform(key, (m.nb_edges,)) * 20 - 10)
    m.faces_params = jnp.asarray([BENCH_TARGET_AREA for _ in range(m.nb_faces)])
    return m


def run_solver_benchmark(name, solver, max_iter,
                         tolerance=BENCH_TOLERANCE, patience=BENCH_PATIENCE):
    """Run the full 25-division sequence with `solver` as inner solver, `max_iter`
    inner iterations per relaxation, and the given early-stopping (tolerance,
    patience). Returns (elapsed_seconds, initial_mesh, final_mesh, params_dict).
    cell_rng is re-seeded to rng(0) so every solver divides the same cells along
    the same axes."""
    m0 = make_bench_mesh()
    m0_snapshot = BoundedMesh.copy_mesh(m0)
    t0 = time.perf_counter()
    m_final, _ = run_n_divisions(
        m0, n_divisions=BENCH_N_DIV,
        cell_rng=np.random.default_rng(0),
        inner_solver=solver,
        max_opt_iter=max_iter,
        tolerance=tolerance,
        patience=patience,
    )
    elapsed = time.perf_counter() - t0
    params = dict(max_iter=max_iter, tolerance=tolerance, patience=patience)
    met = mesh_metrics(m_final)
    print(f"[{name}] {elapsed:.2f} s, {max_iter} it/relax, tol={tolerance:g}, "
          f"patience={patience}  ->  {m_final.nb_faces} cells  "
          f"|  E={met['E']:.3f} (area {met['E_area']:.3f} + tension {met['E_tension']:.3f})")
    return elapsed, m0_snapshot, m_final, params


# Collected per-solver results: name -> (elapsed_s, initial_mesh, final_mesh, params).
bench_results = {}

In [ ]:
# --- SGD -------------------------------------------------------------------------
# Per-solver iteration budget + early-stopping criterion (edit freely).
SGD_MAX_ITER  = 2000
SGD_TOLERANCE = 1e-6
SGD_PATIENCE  = 50
bench_results["SGD"] = run_solver_benchmark(
    "SGD", optax.sgd(learning_rate=1e-3), SGD_MAX_ITER,
    tolerance=SGD_TOLERANCE, patience=SGD_PATIENCE)
sgd_time = bench_results["SGD"][0]
print(f"SGD total time: {sgd_time:.2f} s")

In [ ]:
# --- Adam ------------------------------------------------------------------------
ADAM_MAX_ITER  = 2000
ADAM_TOLERANCE = 1e-6
ADAM_PATIENCE  = 50
bench_results["Adam"] = run_solver_benchmark(
    "Adam", optax.adam(learning_rate=1e-3), ADAM_MAX_ITER,
    tolerance=ADAM_TOLERANCE, patience=ADAM_PATIENCE)
adam_time = bench_results["Adam"][0]
print(f"Adam total time: {adam_time:.2f} s")

In [ ]:
# --- CG (nonlinear conjugate gradient) -------------------------------------------
# Line-search solvers usually need far fewer iterations than plain SGD/Adam.
CG_MAX_ITER  = 500
CG_TOLERANCE = 1e-4
CG_PATIENCE  = 5
bench_results["CG"] = run_solver_benchmark(
    "CG", nonlinear_cg(restart_every=10), CG_MAX_ITER,
    tolerance=CG_TOLERANCE, patience=CG_PATIENCE)
cg_time = bench_results["CG"][0]
print(f"CG total time: {cg_time:.2f} s")

In [ ]:
# --- L-BFGS (requires optax >= 0.2.5; see note above) ----------------------------
LBFGS_MAX_ITER  = 200
LBFGS_TOLERANCE = 1e-6
LBFGS_PATIENCE  = 50
bench_results["L-BFGS"] = run_solver_benchmark(
    "L-BFGS", optax.lbfgs(), LBFGS_MAX_ITER,
    tolerance=LBFGS_TOLERANCE, patience=LBFGS_PATIENCE)
lbfgs_time = bench_results["L-BFGS"][0]
print(f"L-BFGS total time: {lbfgs_time:.2f} s")

In [ ]:
# Per-solver initial vs final mesh. One row per solver; left = initial 4-cell mesh,
# right = final mesh after 25 divisions. Edge colour = tension, cell colour = area.
# The final-mesh title reports the global energy and its area/tension split, the
# mean area and mean tension, the per-solver settings, and the wall-clock time.
from visualization import _draw_mesh_on_ax, _add_cell_labels, AREA_SCALE, TENSION_SCALE
import matplotlib
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

sm_tension = ScalarMappable(Normalize(*TENSION_SCALE), matplotlib.colormaps["coolwarm"])
sm_area    = ScalarMappable(Normalize(*AREA_SCALE),    matplotlib.colormaps["cividis"])


def _row_colorbars(fig, row_axes):
    cbt = fig.colorbar(sm_tension, ax=row_axes, fraction=0.035, pad=0.03, aspect=15)
    cba = fig.colorbar(sm_area,    ax=row_axes, fraction=0.035, pad=0.07, aspect=15)
    for cb, lbl in ((cbt, "tension"), (cba, "Area of cell")):
        cb.ax.yaxis.set_label_position("left")
        cb.set_label(lbl)


def _fit_axes_to_mesh(ax, mesh, margin=0.3):
    """Override the default width/height-based limits set by _draw_mesh_on_ax so the
    WHOLE tissue is visible: after 25 divisions the mesh grows past its initial
    width/height, so we fit the axes to the actual vertex extent (+ a small margin)."""
    v = np.asarray(mesh.vertices)
    xmin, ymin = v.min(axis=0); xmax, ymax = v.max(axis=0)
    ax.set_xlim(xmin - margin, xmax + margin)
    ax.set_ylim(ymin - margin, ymax + margin)


# Keep a stable solver order regardless of dict insertion.
solver_order = [n for n in ["SGD", "Adam", "CG", "L-BFGS"] if n in bench_results]
nrows = len(solver_order)

fig, axes = plt.subplots(nrows, 2, figsize=(17, 5.5 * nrows))
if nrows == 1:
    axes = axes[None, :]
fig.suptitle(f"Inner-solver benchmark — 4-cell mesh, {BENCH_N_DIV} divisions  "
             "|  left: initial  ·  right: final  (edge=tension, cell=area)", fontsize=13)

for row, name in enumerate(solver_order):
    elapsed, m_init, m_final, params = bench_results[name]
    mi = mesh_metrics(m_init)
    mf = mesh_metrics(m_final)
    settings = (f"{params['max_iter']} it/relax, tol={params['tolerance']:g}, "
                f"patience={params['patience']}, {elapsed:.1f} s")
    panels = [
        (m_init, mi,
         f"{name} — initial ({m_init.nb_faces} cells)\n"
         f"E={mi['E']:.2f} (area {mi['E_area']:.2f} + tension {mi['E_tension']:.2f})  ·  "
         f"areā={mi['mean_area']:.3f}  tension̄={mi['mean_tension']:.3f}"),
        (m_final, mf,
         f"{name} — final ({m_final.nb_faces} cells, {settings})\n"
         f"E={mf['E']:.2f} (area {mf['E_area']:.2f} + tension {mf['E_tension']:.2f})  ·  "
         f"areā={mf['mean_area']:.3f}  tension̄={mf['mean_tension']:.3f}"),
    ]
    for col, (snap, met, title) in enumerate(panels):
        ax = axes[row, col]
        _draw_mesh_on_ax(snap, ax, AREA_SCALE, TENSION_SCALE)
        _fit_axes_to_mesh(ax, snap)   # show the full contour (mesh grows past width/height)
        _add_cell_labels(ax, snap)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    _row_colorbars(fig, axes[row, :])

plt.show()

In [ ]:
# Timing + quality summary — ranking from fastest to slowest over the 25-division run.
# Reports each solver's settings (iterations, tolerance, patience), wall-clock time,
# and the final mesh's global energy: fastest is not always lowest-energy.
ranked = sorted(bench_results.items(), key=lambda kv: kv[1][0])
fastest_name, (fastest_t, _, _, _) = ranked[0]

print(f"Inner-solver benchmark — 4-cell mesh, {BENCH_N_DIV} consecutive divisions")
print("-" * 92)
print(f"{'solver':8}  {'it/relax':>8}  {'tol':>7}  {'patience':>8}  {'time (s)':>9}  "
      f"{'vs fast':>8}  {'E_final':>9}")
for name, (elapsed, _, m_final, params) in ranked:
    mf = mesh_metrics(m_final)
    print(f"{name:8}  {params['max_iter']:>8}  {params['tolerance']:>7g}  "
          f"{params['patience']:>8}  {elapsed:>9.2f}  {elapsed / fastest_t:>7.2f}x  "
          f"{mf['E']:>9.2f}")
print("-" * 92)
print(f"Fastest: {fastest_name} ({fastest_t:.2f} s)")

# Two-panel summary: wall-clock time (left) and final global energy (right).
names = [n for n, _ in ranked]
times = [v[0] for _, v in ranked]
energies = [mesh_metrics(v[2])['E'] for _, v in ranked]
colors = ["#2a9d8f", "#457b9d", "#e9c46a", "#e76f51"][:len(names)]

fig, (ax_t, ax_e) = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle(f"Inner-solver benchmark — {BENCH_N_DIV} divisions on a 4-cell mesh", fontsize=13)

bars_t = ax_t.bar(names, times, color=colors)
ax_t.set_ylabel("wall-clock time (s)")
ax_t.set_title("Speed (lower is faster)")
for b, t in zip(bars_t, times):
    ax_t.text(b.get_x() + b.get_width() / 2, t, f"{t:.1f}s", ha="center", va="bottom", fontsize=9)

bars_e = ax_e.bar(names, energies, color=colors)
ax_e.set_ylabel("final global energy  E")
ax_e.set_title("Solution quality (lower energy = better relaxation)")
for b, e in zip(bars_e, energies):
    ax_e.text(b.get_x() + b.get_width() / 2, e, f"{e:.1f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## Parallel (simultaneous) divisions — N waves of M divisions

Instead of relaxing the tissue after *every* division (`run_n_divisions`), this
section divides **several cells at once per wave** and relaxes the tissue **once
per wave** (`run_n_divisions_parallel`). With `N_WAVES` waves of `N_PARALLEL`
simultaneous cuts, the tissue undergoes `N_WAVES * N_PARALLEL` divisions but only
`N_WAVES` relaxations — the relaxation being the dominant cost, this is ~2x
faster for the same number of divisions.

This is a **synchronous-mitosis** model: the cells of a wave are cut on the same
(un-relaxed) mesh, so the result differs from the sequential one — it is not just
a speed-up of the identical computation.

Set the two knobs below: `N_WAVES` (number of relaxation steps) and `N_PARALLEL`
(cells divided simultaneously in each wave).

In [ ]:
import importlib
import spindle_analysis; importlib.reload(spindle_analysis)
from spindle_analysis import run_n_divisions_parallel

# Requires the inner-solver benchmark setup cell to have run first (it defines
# the imports, CUDA_ROOT, and make_bench_mesh()).

# === knobs: e.g. 5 waves of 5 simultaneous divisions = 25 divisions, 5 relaxations
N_WAVES    = 5
N_PARALLEL = 5

# Start from a fresh 4-cell mesh (same as the solver benchmark).
mesh_par = make_bench_mesh()
mesh_final_par, wave_snapshots = run_n_divisions_parallel(
    mesh_par, n_waves=N_WAVES, n_parallel=N_PARALLEL,
    cell_rng=np.random.default_rng(0),
)
print(f"After {N_WAVES} waves x {N_PARALLEL} divisions: {mesh_final_par.nb_faces} cells, "
      f"{mesh_final_par.nb_edges} edges  ({N_WAVES} relaxations)")

In [ ]:
from visualization import _draw_mesh_on_ax, _add_cell_labels, AREA_SCALE, TENSION_SCALE
import matplotlib
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

sm_tension = ScalarMappable(Normalize(*TENSION_SCALE), matplotlib.colormaps["coolwarm"])
sm_area    = ScalarMappable(Normalize(*AREA_SCALE),    matplotlib.colormaps["cividis"])


def _row_colorbars(fig, row_axes):
    cbt = fig.colorbar(sm_tension, ax=row_axes, fraction=0.035, pad=0.03, aspect=15)
    cba = fig.colorbar(sm_area,    ax=row_axes, fraction=0.035, pad=0.07, aspect=15)
    for cb, lbl in ((cbt, "tension"), (cba, "Area of cell")):
        cb.ax.yaxis.set_label_position("left")
        cb.set_label(lbl)


def _fit_axes_to_mesh(ax, m, margin=0.3):
    v = np.asarray(m.vertices)
    xmin, ymin = v.min(axis=0); xmax, ymax = v.max(axis=0)
    ax.set_xlim(xmin - margin, xmax + margin)
    ax.set_ylim(ymin - margin, ymax + margin)


fig, axes = plt.subplots(N_WAVES, 2, figsize=(16, 5 * N_WAVES))
if N_WAVES == 1:
    axes = axes[None, :]
fig.suptitle(f"{N_WAVES} waves x {N_PARALLEL} simultaneous divisions  "
             "|  left: just after the wave's cuts  .  right: after relaxation",
             fontsize=13)

for w, (snap_post, snap_opt) in enumerate(wave_snapshots, start=1):
    row = w - 1
    for col, (snap, label) in enumerate([
        (snap_post, f"Wave {w} - after cuts ({snap_post.nb_faces} cells)"),
        (snap_opt,  f"Wave {w} - after relaxation ({snap_opt.nb_faces} cells)"),
    ]):
        ax = axes[row, col]
        ta = float(jnp.sum(snap.get_area(jnp.arange(snap.nb_faces))))
        _draw_mesh_on_ax(snap, ax, AREA_SCALE, TENSION_SCALE)
        _fit_axes_to_mesh(ax, snap)
        _add_cell_labels(ax, snap)
        ax.set_title(f"{label} - area: {ta:.3f}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    _row_colorbars(fig, axes[row, :])

plt.show()

## Animation — spindle set-up, parallel divisions, progressive relaxation

Each wave now has three phases, all animated:
1. **Spindle** — for every cell to divide, the exact spindle of the chosen method eases
   deterministically from a perturbed start (offset centre, random axis); the
   overlay shows the two **centrosomes** (green), the **centre** (red x) and the
   **cleavage line** (yellow dashed, ⟂ to the spindle axis).
2. **Cut** — every cell is divided at its settled (centre, axis).
3. **Relaxation** — the tissue relaxes once, sampled along the true trajectory.

Rendered to MP4 (ffmpeg via imageio-ffmpeg) and shown inline. Knobs in the run
cell: N_WAVES, N_PARALLEL, METHOD, SPINDLE_FRAMES, RELAX_FRAMES, START_OFFSET,
INNER_SOLVER. Requires the inner-solver benchmark setup cell to have run first.

In [ ]:
import importlib
import spindle_analysis; importlib.reload(spindle_analysis)
from spindle_analysis import run_n_divisions_parallel_animated

# === knobs: animate parallel (wave) divisions with a pre-division spindle phase ===
N_WAVES        = 5      # number of waves
N_PARALLEL     = 5      # cells divided simultaneously per wave
METHOD         = "newton"   # division method: "newton" | "thery" | "polarity" | "edge_score"
SPINDLE_FRAMES = 20     # frames of the (deterministic) spindle set-up before each cut
RELAX_FRAMES   = 10     # frames sampled along each relaxation
START_OFFSET   = 0.3    # initial spindle-centre offset (fraction of cell radius)
ANIM_MAX_ITER  = 1000   # max inner iterations per relaxation
INNER_SOLVER   = optax.sgd(learning_rate=1e-3)
#   optax.sgd(learning_rate=1e-3)        # plain gradient descent
#   optax.adam(learning_rate=1e-3)       # adaptive first-order
#   nonlinear_cg(restart_every=10)       # conjugate gradient (Polak-Ribiere + line search)
#   optax.lbfgs()                        # limited-memory quasi-Newton (line search)
# Method extras (used by the matching method):
POLARITY_STRENGTH = 1.0   # 'polarity': how strongly the tension polarity biases the axis
EDGE_BIAS_KAPPA   = 4.0   # 'edge_score': anchor on the Newton axis
EDGE_BIAS_BETA    = 2.0   # 'edge_score': edge-score bias strength
# ==================================================================================

mesh_anim = make_bench_mesh()   # fresh 4-cell mesh (from the benchmark setup cell)
mesh_anim_final, anim_frames = run_n_divisions_parallel_animated(
    mesh_anim, n_waves=N_WAVES, n_parallel=N_PARALLEL, method=METHOD,
    spindle_frames=SPINDLE_FRAMES, relax_frames=RELAX_FRAMES,
    start_offset=START_OFFSET,
    polarity_strength=POLARITY_STRENGTH,
    edge_bias_kappa=EDGE_BIAS_KAPPA, edge_bias_beta=EDGE_BIAS_BETA,
    cell_rng=np.random.default_rng(0), spindle_rng=np.random.default_rng(1),
    edge_bias_rng=np.random.default_rng(2),
    inner_solver=INNER_SOLVER, max_opt_iter=ANIM_MAX_ITER,
)
n_spindle = sum(1 for f in anim_frames if f["kind"] == "spindle")
print(f"method={METHOD}: {len(anim_frames)} frames "
      f"({n_spindle} spindle + cuts + relax), final {mesh_anim_final.nb_faces} cells")

In [ ]:
import matplotlib
import imageio_ffmpeg
matplotlib.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from visualization import _draw_mesh_on_ax, _add_cell_labels, AREA_SCALE, TENSION_SCALE


# Fixed axes limits: large enough from the start to contain EVERY frame's mesh
# (plus the spindle overlays), so the frame never jumps as the tissue grows.
def _global_limits(frames, margin=0.4):
    xs_min = ys_min = np.inf
    xs_max = ys_max = -np.inf
    for fr in frames:
        v = np.asarray(fr['mesh'].vertices)
        xs_min = min(xs_min, v[:, 0].min()); xs_max = max(xs_max, v[:, 0].max())
        ys_min = min(ys_min, v[:, 1].min()); ys_max = max(ys_max, v[:, 1].max())
        for (c, alpha, d) in fr.get('spindles', []):
            u = np.array([np.cos(alpha), np.sin(alpha)]); cut = np.array([-u[1], u[0]])
            for p in (c + 4 * d * cut, c - 4 * d * cut, c + d * u, c - d * u):
                xs_min = min(xs_min, p[0]); xs_max = max(xs_max, p[0])
                ys_min = min(ys_min, p[1]); ys_max = max(ys_max, p[1])
    return (xs_min - margin, xs_max + margin, ys_min - margin, ys_max + margin)


XMIN, XMAX, YMIN, YMAX = _global_limits(anim_frames)


def _draw_spindles(ax, spindles):
    for (c, alpha, d) in spindles:
        u = np.array([np.cos(alpha), np.sin(alpha)])
        cut = np.array([-u[1], u[0]])
        c1, c2 = c + d * u, c - d * u
        L = 4 * d
        p0, p1 = c - L * cut, c + L * cut
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], '--', color='yellow', lw=1.5, zorder=6)
        ax.plot([c1[0], c2[0]], [c1[1], c2[1]], '-', color='lime', lw=1.2, zorder=6)
        ax.scatter([c1[0], c2[0]], [c1[1], c2[1]], s=45, color='lime',
                   edgecolors='black', zorder=7)
        ax.scatter([c[0]], [c[1]], marker='x', s=40, color='red', zorder=7)


fig, ax = plt.subplots(figsize=(7, 7))


def _render(frame_idx):
    ax.clear()
    fr = anim_frames[frame_idx]
    m = fr['mesh']
    _draw_mesh_on_ax(m, ax, AREA_SCALE, TENSION_SCALE)
    ax.set_xlim(XMIN, XMAX); ax.set_ylim(YMIN, YMAX)   # fixed frame
    ax.set_aspect('equal')
    _add_cell_labels(ax, m)
    if fr['kind'] == 'spindle':
        _draw_spindles(ax, fr['spindles'])
        tag = f"spindle {fr['sub']}/{SPINDLE_FRAMES}"
    elif fr['kind'] == 'division':
        tag = "CUT"
    else:
        tag = f"relax {fr['sub']}/{RELAX_FRAMES}"
    ax.set_title(f"Wave {fr['division'] + 1}/{N_WAVES} - {tag}  "
                 f"({m.nb_faces} cells)  .  E = {fr['energy']:.2f}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    return ax,


anim = FuncAnimation(fig, _render, frames=len(anim_frames), interval=150, blit=False)
anim.save("mitotic_divisions.mp4", writer="ffmpeg", fps=8, dpi=110)
plt.close(fig)
print("saved mitotic_divisions.mp4")
# HTML(anim.to_html5_video())